In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
import os
import json
import pickle
import random
import hashlib
import shutil
import platform
import datetime
import importlib.metadata as im

import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

SEEDS = [42, 123, 2024]

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")
SEQ_DIR = BASE_PROJECT / "processed_intra_sequence_features_plus_base"
OUT_DIR = BASE_PROJECT / "results_lodo_sequence_cross_attention_plus_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]
LABELS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]
ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL_TO_ID = {label: i for i, label in ID_TO_LABEL.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)
print("BASE_PROJECT:", BASE_PROJECT)
print("SEQ_DIR:", SEQ_DIR)
print("OUT_DIR:", OUT_DIR)

for ds in DATASETS:
    print(ds, (SEQ_DIR / ds).exists())



DEVICE: cuda
BASE_PROJECT: /content/drive/MyDrive/New Jurnal Cross
SEQ_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base
OUT_DIR: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base
emodb True
ravdess True
resd True


In [3]:
def pkg_ver(name):
    try:
        return im.version(name)
    except Exception:
        return "not installed"


def sha256_of_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


# Save the training/runtime environment for reproducibility.
training_env_manifest = {
    "script": "11_train_lodo_sequence_cross_attention.ipynb",
    "base_project": str(BASE_PROJECT),
    "seq_dir": str(SEQ_DIR),
    "out_dir": str(OUT_DIR),
    "created_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "python_version": platform.python_version(),
    "torch_version": pkg_ver("torch"),
    "numpy_version": pkg_ver("numpy"),
    "pandas_version": pkg_ver("pandas"),
    "scikit_learn_version": pkg_ver("scikit-learn"),
}

with open(OUT_DIR / "sequence_training_environment_manifest.json", "w") as f:
    json.dump(training_env_manifest, f, indent=2)

print(json.dumps(training_env_manifest, indent=2))


# Hash cached sequence feature files used by this LODO notebook.
feature_manifest_rows = []
input_suffixes = {".npy", ".csv", ".pkl", ".json"}

if SEQ_DIR.exists():
    for path in sorted(SEQ_DIR.rglob("*")):
        if path.is_file() and path.suffix in input_suffixes:
            feature_manifest_rows.append({
                "relative_path": str(path.relative_to(BASE_PROJECT)),
                "absolute_path": str(path),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_of_file(path),
            })

feature_manifest = pd.DataFrame(feature_manifest_rows)
feature_manifest.to_csv(OUT_DIR / "sequence_input_feature_file_manifest_sha256.csv", index=False)
print("Saved:", OUT_DIR / "sequence_input_feature_file_manifest_sha256.csv")
print("Number of sequence feature/cache files hashed:", len(feature_manifest))

display(feature_manifest.head())


# Copy emotion2vec extraction environment manifest if notebook 08b produced it.
for candidate in [
    SEQ_DIR / "environment_manifest.json",
    SEQ_DIR / "sequence_environment_manifest.json",
    SEQ_DIR / "emotion2vec_sequence_environment_manifest.json",
]:
    if candidate.exists():
        shutil.copy2(candidate, OUT_DIR / "emotion2vec_sequence_extraction_environment_manifest.json")
        print("Copied sequence extraction manifest:", candidate)
        break
else:
    print("Warning: no sequence extraction environment_manifest.json found in", SEQ_DIR)



{
  "script": "11_train_lodo_sequence_cross_attention.ipynb",
  "base_project": "/content/drive/MyDrive/New Jurnal Cross",
  "seq_dir": "/content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base",
  "out_dir": "/content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base",
  "created_utc": "2026-08-08T21:38:19.370480Z",
  "python_version": "3.12.13",
  "torch_version": "2.11.0+cu128",
  "numpy_version": "2.0.2",
  "pandas_version": "2.2.2",
  "scikit_learn_version": "1.6.1"
}


/tmp/ipykernel_7756/3269942982.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc": datetime.datetime.utcnow().isoformat() + "Z",


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/sequence_input_feature_file_manifest_sha256.csv
Number of sequence feature/cache files hashed: 60


,relative_path,absolute_path,size_bytes,sha256
0,processed_intra_sequence_features_plus_base/em...,/content/drive/MyDrive/New Jurnal Cross/proces...,91545728,e4f2d821e110ecbea46d670ca7ddfe653b94cb00f512fd...
1,processed_intra_sequence_features_plus_base/em...,/content/drive/MyDrive/New Jurnal Cross/proces...,305971328,008010ade43f83a69de18d086514537322f810a341a2f0...
2,processed_intra_sequence_features_plus_base/em...,/content/drive/MyDrive/New Jurnal Cross/proces...,43622528,17e56db66e550d1d9c971c418a4edb7c84ec925d7c95e3...
3,processed_intra_sequence_features_plus_base/em...,/content/drive/MyDrive/New Jurnal Cross/proces...,5125728,666abf5ed8b746f4ff6bbc654249e4a96daa9d9dc9cdbc...
4,processed_intra_sequence_features_plus_base/em...,/content/drive/MyDrive/New Jurnal Cross/proces...,17131328,c25685dd2fc7ba9dfb2358d510c6790c3f96dc6955e97b...


Copied sequence extraction manifest: /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base/sequence_environment_manifest.json


In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def make_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=LABELS,
        labels=list(range(len(LABELS))),
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).transpose()


def save_confusion_matrix_csv(cm, out_path):
    df_cm = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    df_cm.to_csv(out_path, index=True)


def compute_class_weights(y_train, n_classes=6):
    counts = np.bincount(y_train, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / (n_classes * np.maximum(counts, 1.0))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def count_trainable_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))



In [5]:
def inverse_transform_sequence(X_scaled, scaler):
    """Recover raw sequence values from an intra-corpus scaler.

    X_scaled: [N, T, D]
    """
    N, T, D = X_scaled.shape
    X_raw = scaler.inverse_transform(X_scaled.reshape(-1, D)).reshape(N, T, D)
    return X_raw.astype(np.float32)


def prepare_meta(meta, dataset_name, split_name):
    meta = meta.copy()
    if "dataset" not in meta.columns:
        meta["dataset"] = dataset_name
    if "corpus" not in meta.columns:
        meta["corpus"] = dataset_name
    if "original_split" not in meta.columns:
        meta["original_split"] = split_name
    return meta


def load_one_sequence_dataset_raw(dataset_name):
    ds_dir = SEQ_DIR / dataset_name

    X_e2v_train_scaled = np.load(ds_dir / "X_e2v_seq_train.npy").astype(np.float32)
    X_e2v_val_scaled = np.load(ds_dir / "X_e2v_seq_val.npy").astype(np.float32)
    X_e2v_test_scaled = np.load(ds_dir / "X_e2v_seq_test.npy").astype(np.float32)

    X_hc_train_scaled = np.load(ds_dir / "X_hc_seq_train.npy").astype(np.float32)
    X_hc_val_scaled = np.load(ds_dir / "X_hc_seq_val.npy").astype(np.float32)
    X_hc_test_scaled = np.load(ds_dir / "X_hc_seq_test.npy").astype(np.float32)

    mask_train = np.load(ds_dir / "mask_train.npy").astype(np.float32)
    mask_val = np.load(ds_dir / "mask_val.npy").astype(np.float32)
    mask_test = np.load(ds_dir / "mask_test.npy").astype(np.float32)

    y_train = np.load(ds_dir / "y_train.npy").astype(np.int64)
    y_val = np.load(ds_dir / "y_val.npy").astype(np.int64)
    y_test = np.load(ds_dir / "y_test.npy").astype(np.int64)

    meta_train = prepare_meta(pd.read_csv(ds_dir / "meta_train.csv"), dataset_name, "train")
    meta_val = prepare_meta(pd.read_csv(ds_dir / "meta_val.csv"), dataset_name, "val")
    meta_test = prepare_meta(pd.read_csv(ds_dir / "meta_test.csv"), dataset_name, "test")

    with open(ds_dir / "scaler_e2v_seq.pkl", "rb") as f:
        scaler_e2v_intra = pickle.load(f)

    with open(ds_dir / "scaler_hc_seq.pkl", "rb") as f:
        scaler_hc_intra = pickle.load(f)

    X_e2v_train = inverse_transform_sequence(X_e2v_train_scaled, scaler_e2v_intra)
    X_e2v_val = inverse_transform_sequence(X_e2v_val_scaled, scaler_e2v_intra)
    X_e2v_test = inverse_transform_sequence(X_e2v_test_scaled, scaler_e2v_intra)

    X_hc_train = inverse_transform_sequence(X_hc_train_scaled, scaler_hc_intra)
    X_hc_val = inverse_transform_sequence(X_hc_val_scaled, scaler_hc_intra)
    X_hc_test = inverse_transform_sequence(X_hc_test_scaled, scaler_hc_intra)

    return {
        "X_e2v_train": X_e2v_train,
        "X_e2v_val": X_e2v_val,
        "X_e2v_test": X_e2v_test,
        "X_hc_train": X_hc_train,
        "X_hc_val": X_hc_val,
        "X_hc_test": X_hc_test,
        "mask_train": mask_train,
        "mask_val": mask_val,
        "mask_test": mask_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,
    }


seq_cache = {}
for ds in DATASETS:
    seq_cache[ds] = load_one_sequence_dataset_raw(ds)
    print("=" * 80)
    print(ds.upper())
    print("E2V train:", seq_cache[ds]["X_e2v_train"].shape)
    print("HC train :", seq_cache[ds]["X_hc_train"].shape)
    print("mask     :", seq_cache[ds]["mask_train"].shape)
    print("y        :", seq_cache[ds]["y_train"].shape)
    print("NaN E2V  :", np.isnan(seq_cache[ds]["X_e2v_train"]).any())
    print("NaN HC   :", np.isnan(seq_cache[ds]["X_hc_train"]).any())



EMODB
E2V train: (498, 200, 768)
HC train : (498, 200, 43)
mask     : (498, 200)
y        : (498,)
NaN E2V  : False
NaN HC   : False
RAVDESS
E2V train: (704, 200, 768)
HC train : (704, 200, 43)
mask     : (704, 200)
y        : (704,)
NaN E2V  : False
NaN HC   : False
RESD
E2V train: (873, 200, 768)
HC train : (873, 200, 43)
mask     : (873, 200)
y        : (873,)
NaN E2V  : False
NaN HC   : False


In [6]:
def fit_transform_sequence_scaler(X_train, X_val, X_test, mask_train, mask_val, mask_test):
    """Fit scaler only on valid source-training frames, then transform train/val/test.

    Padded or invalid frames are set to 0.0 after transformation so that artificial
    padding values do not contribute to the classifier. The target test data are
    never used in scaler fitting.
    """
    _, _, D = X_train.shape
    valid_train = mask_train.astype(bool)
    if valid_train.sum() == 0:
        raise ValueError("No valid training frames found for scaler fitting.")

    scaler = StandardScaler()
    scaler.fit(X_train[valid_train].reshape(-1, D))

    def transform_and_zero_invalid(X, mask):
        X_s = scaler.transform(X.reshape(-1, D)).reshape(X.shape).astype(np.float32)
        X_s[~mask.astype(bool)] = 0.0
        return X_s

    X_train_s = transform_and_zero_invalid(X_train, mask_train)
    X_val_s = transform_and_zero_invalid(X_val, mask_val)
    X_test_s = transform_and_zero_invalid(X_test, mask_test)

    return X_train_s, X_val_s, X_test_s, scaler



In [7]:
def concat_parts(parts, axis=0):
    return np.concatenate(parts, axis=axis)


def make_lodo_sequence_fold(test_dataset):
    source_datasets = [ds for ds in DATASETS if ds != test_dataset]

    X_e2v_train = concat_parts(
        [seq_cache[ds]["X_e2v_train"] for ds in source_datasets] +
        [seq_cache[ds]["X_e2v_test"] for ds in source_datasets]
    )
    X_hc_train = concat_parts(
        [seq_cache[ds]["X_hc_train"] for ds in source_datasets] +
        [seq_cache[ds]["X_hc_test"] for ds in source_datasets]
    )
    mask_train = concat_parts(
        [seq_cache[ds]["mask_train"] for ds in source_datasets] +
        [seq_cache[ds]["mask_test"] for ds in source_datasets]
    )
    y_train = concat_parts(
        [seq_cache[ds]["y_train"] for ds in source_datasets] +
        [seq_cache[ds]["y_test"] for ds in source_datasets]
    )
    meta_train = pd.concat(
        [seq_cache[ds]["meta_train"] for ds in source_datasets] +
        [seq_cache[ds]["meta_test"] for ds in source_datasets],
        ignore_index=True,
    )

    X_e2v_val = concat_parts([seq_cache[ds]["X_e2v_val"] for ds in source_datasets])
    X_hc_val = concat_parts([seq_cache[ds]["X_hc_val"] for ds in source_datasets])
    mask_val = concat_parts([seq_cache[ds]["mask_val"] for ds in source_datasets])
    y_val = concat_parts([seq_cache[ds]["y_val"] for ds in source_datasets])
    meta_val = pd.concat(
        [seq_cache[ds]["meta_val"] for ds in source_datasets],
        ignore_index=True,
    )

    target = seq_cache[test_dataset]
    X_e2v_test = concat_parts([
        target["X_e2v_train"],
        target["X_e2v_val"],
        target["X_e2v_test"],
    ])
    X_hc_test = concat_parts([
        target["X_hc_train"],
        target["X_hc_val"],
        target["X_hc_test"],
    ])
    mask_test = concat_parts([
        target["mask_train"],
        target["mask_val"],
        target["mask_test"],
    ])
    y_test = concat_parts([
        target["y_train"],
        target["y_val"],
        target["y_test"],
    ])
    meta_test = pd.concat(
        [target["meta_train"], target["meta_val"], target["meta_test"]],
        ignore_index=True,
    )

    X_e2v_train_s, X_e2v_val_s, X_e2v_test_s, scaler_e2v = fit_transform_sequence_scaler(
        X_e2v_train, X_e2v_val, X_e2v_test, mask_train, mask_val, mask_test
    )
    X_hc_train_s, X_hc_val_s, X_hc_test_s, scaler_hc = fit_transform_sequence_scaler(
        X_hc_train, X_hc_val, X_hc_test, mask_train, mask_val, mask_test
    )

    return {
        "test_dataset": test_dataset,
        "source_datasets": source_datasets,
        "X_e2v_train": X_e2v_train_s,
        "X_e2v_val": X_e2v_val_s,
        "X_e2v_test": X_e2v_test_s,
        "X_hc_train": X_hc_train_s,
        "X_hc_val": X_hc_val_s,
        "X_hc_test": X_hc_test_s,
        "mask_train": mask_train,
        "mask_val": mask_val,
        "mask_test": mask_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,
        "scaler_e2v": scaler_e2v,
        "scaler_hc": scaler_hc,
        "e2v_dim": X_e2v_train_s.shape[-1],
        "hc_dim": X_hc_train_s.shape[-1],
        "target_frames": X_e2v_train_s.shape[1],
    }


for test_ds in DATASETS:
    fold = make_lodo_sequence_fold(test_ds)
    print("=" * 90)
    print("HELD-OUT:", test_ds.upper())
    print("SOURCE  :", fold["source_datasets"])
    print("Train E2V:", fold["X_e2v_train"].shape)
    print("Train HC :", fold["X_hc_train"].shape)
    print("Val E2V  :", fold["X_e2v_val"].shape)
    print("Test E2V :", fold["X_e2v_test"].shape)
    print("Valid frame ratio train/val/test:",
          round(float(fold["mask_train"].mean()), 4),
          round(float(fold["mask_val"].mean()), 4),
          round(float(fold["mask_test"].mean()), 4))
    print("Test labels:")
    print(pd.Series(fold["y_test"]).value_counts().sort_index().rename(index=ID_TO_LABEL))



HELD-OUT: EMODB
SOURCE  : ['ravdess', 'resd']
Train E2V: (1892, 200, 768)
Train HC : (1892, 200, 43)
Val E2V  : (362, 200, 768)
Test E2V : (718, 200, 768)
Valid frame ratio train/val/test: 1.0 1.0 1.0
Test labels:
angry      140
disgust    106
fear       123
happy      118
neutral    106
sad        125
Name: count, dtype: int64
HELD-OUT: RAVDESS
SOURCE  : ['emodb', 'resd']
Train E2V: (1659, 200, 768)
Train HC : (1659, 200, 43)
Val E2V  : (257, 200, 768)
Test E2V : (1056, 200, 768)
Valid frame ratio train/val/test: 1.0 1.0 1.0
Test labels:
angry      192
disgust    192
fear       192
happy      192
neutral     96
sad        192
Name: count, dtype: int64
HELD-OUT: RESD
SOURCE  : ['emodb', 'ravdess']
Train E2V: (1527, 200, 768)
Train HC : (1527, 200, 43)
Val E2V  : (247, 200, 768)
Test E2V : (1198, 200, 768)
Valid frame ratio train/val/test: 1.0 1.0 1.0
Test labels:
angry      219
disgust    185
fear       223
happy      218
neutral    191
sad        162
Name: count, dtype: int64


In [8]:
def infer_file_key(meta_df):
    candidates = [
        "file_id", "utt_id", "utterance_id", "audio_id", "id",
        "path", "filepath", "file_path", "wav_path", "audio_path",
        "filename", "file_name", "basename",
    ]
    for col in candidates:
        if col in meta_df.columns:
            return meta_df[col].astype(str)
    return pd.Series([f"row_{i}" for i in range(len(meta_df))])


def make_split_manifest_for_fold(fold):
    frames = []
    for role, meta in [
        ("source_train", fold["meta_train"]),
        ("source_validation", fold["meta_val"]),
        ("target_test", fold["meta_test"]),
    ]:
        m = meta.copy()
        m.insert(0, "fold", f"heldout_{fold['test_dataset']}")
        m.insert(1, "heldout_dataset", fold["test_dataset"])
        m.insert(2, "source_datasets", "+".join(fold["source_datasets"]))
        m.insert(3, "role", role)
        m["file_key"] = infer_file_key(m)
        frames.append(m)
    return pd.concat(frames, ignore_index=True)


manifest_frames = []
summary_rows = []
overlap_rows = []

for test_ds in DATASETS:
    fold = make_lodo_sequence_fold(test_ds)
    manifest = make_split_manifest_for_fold(fold)
    manifest_frames.append(manifest)

    summary = (
        manifest.groupby(["fold", "heldout_dataset", "role", "dataset", "original_split"])
        .size()
        .reset_index(name="n")
    )
    summary_rows.append(summary)

    # Pairwise overlap check by file_key within each LODO fold.
    role_sets = {
        role: set(manifest.loc[manifest["role"] == role, "file_key"].astype(str))
        for role in ["source_train", "source_validation", "target_test"]
    }
    for a, b in [("source_train", "source_validation"), ("source_train", "target_test"), ("source_validation", "target_test")]:
        overlap = sorted(role_sets[a].intersection(role_sets[b]))
        overlap_rows.append({
            "fold": f"heldout_{test_ds}",
            "role_a": a,
            "role_b": b,
            "n_overlap_file_keys": len(overlap),
            "example_overlap_file_keys": ";".join(overlap[:10]),
        })

split_manifest = pd.concat(manifest_frames, ignore_index=True)
split_manifest.to_csv(OUT_DIR / "lodo_sequence_split_manifest.csv", index=False)

split_summary = pd.concat(summary_rows, ignore_index=True)
split_summary.to_csv(OUT_DIR / "lodo_sequence_split_summary.csv", index=False)

overlap_check = pd.DataFrame(overlap_rows)
overlap_check.to_csv(OUT_DIR / "lodo_sequence_partition_overlap_check.csv", index=False)

leakage_checklist = pd.DataFrame([
    {"component": "LODO train split", "procedure": "source train + source test only", "target_used_for_fitting_or_selection": "No"},
    {"component": "LODO validation split", "procedure": "source validation only", "target_used_for_fitting_or_selection": "No"},
    {"component": "LODO target test split", "procedure": "all target splits used only for final evaluation", "target_used_for_fitting_or_selection": "No, except final metric computation"},
    {"component": "Sequence feature scaling", "procedure": "StandardScaler fit only on valid source-training frames; validation and target are transform-only", "target_used_for_fitting_or_selection": "No"},
    {"component": "Class weighting", "procedure": "computed from source-training labels only", "target_used_for_fitting_or_selection": "No"},
    {"component": "Early stopping", "procedure": "selected by source-validation Macro-F1", "target_used_for_fitting_or_selection": "No"},
    {"component": "Checkpoint selection", "procedure": "best epoch selected by source-validation Macro-F1 for each seed", "target_used_for_fitting_or_selection": "No"},
    {"component": "Target labels", "procedure": "used only after prediction for final metric calculation", "target_used_for_fitting_or_selection": "Evaluation only"},
])
leakage_checklist.to_csv(OUT_DIR / "lodo_sequence_leakage_prevention_checklist.csv", index=False)

print("Saved split manifest:", OUT_DIR / "lodo_sequence_split_manifest.csv")
print("Saved split summary:", OUT_DIR / "lodo_sequence_split_summary.csv")
print("Saved overlap check:", OUT_DIR / "lodo_sequence_partition_overlap_check.csv")
print("Saved leakage checklist:", OUT_DIR / "lodo_sequence_leakage_prevention_checklist.csv")
display(split_summary)
display(overlap_check)
display(leakage_checklist)



Saved split manifest: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_split_manifest.csv
Saved split summary: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_split_summary.csv
Saved overlap check: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_partition_overlap_check.csv
Saved leakage checklist: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_leakage_prevention_checklist.csv


,fold,heldout_dataset,role,dataset,original_split,n
0,heldout_emodb,emodb,source_train,ravdess,test,176
1,heldout_emodb,emodb,source_train,ravdess,train,704
2,heldout_emodb,emodb,source_train,resd,test,139
3,heldout_emodb,emodb,source_train,resd,train,873
4,heldout_emodb,emodb,source_validation,ravdess,val,176
5,heldout_emodb,emodb,source_validation,resd,val,186
6,heldout_emodb,emodb,target_test,emodb,test,149
7,heldout_emodb,emodb,target_test,emodb,train,498
8,heldout_emodb,emodb,target_test,emodb,val,71
9,heldout_ravdess,ravdess,source_train,emodb,test,149


,fold,role_a,role_b,n_overlap_file_keys,example_overlap_file_keys
0,heldout_emodb,source_train,source_validation,0,
1,heldout_emodb,source_train,target_test,0,
2,heldout_emodb,source_validation,target_test,0,
3,heldout_ravdess,source_train,source_validation,0,
4,heldout_ravdess,source_train,target_test,0,
5,heldout_ravdess,source_validation,target_test,0,
6,heldout_resd,source_train,source_validation,0,
7,heldout_resd,source_train,target_test,0,
8,heldout_resd,source_validation,target_test,0,


,component,procedure,target_used_for_fitting_or_selection
0,LODO train split,source train + source test only,No
1,LODO validation split,source validation only,No
2,LODO target test split,all target splits used only for final evaluation,"No, except final metric computation"
3,Sequence feature scaling,StandardScaler fit only on valid source-traini...,No
4,Class weighting,computed from source-training labels only,No
5,Early stopping,selected by source-validation Macro-F1,No
6,Checkpoint selection,best epoch selected by source-validation Macro...,No
7,Target labels,used only after prediction for final metric ca...,Evaluation only


In [9]:
class SequenceFusionDataset(Dataset):
    def __init__(self, X_e2v, X_hc, mask, y):
        self.X_e2v = torch.tensor(X_e2v, dtype=torch.float32)
        self.X_hc = torch.tensor(X_hc, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_e2v[idx], self.X_hc[idx], self.mask[idx], self.y[idx]


def make_sequence_loader(X_e2v, X_hc, mask, y, batch_size=8, shuffle=False):
    dataset = SequenceFusionDataset(X_e2v, X_hc, mask, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)



In [10]:
class AttentivePooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, x, mask=None):
        scores = self.attn(x).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled, weights


def masked_mean_pool(x, mask):
    mask = mask.unsqueeze(-1)
    x = x * mask
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return x.sum(dim=1) / denom


def masked_max_pool(x, mask):
    mask = mask.unsqueeze(-1)
    x = x.masked_fill(mask == 0, -1e9)
    return x.max(dim=1).values


class SequenceCrossAttentionFusion(nn.Module):
    def __init__(self, e2v_dim=768, hc_dim=43, d_model=128, num_heads=4, num_classes=6, dropout=0.30):
        super().__init__()
        self.e2v_proj = nn.Sequential(nn.Linear(e2v_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.hc_proj = nn.Sequential(nn.Linear(hc_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.e2v_to_hc = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.hc_to_e2v = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm_e = nn.LayerNorm(d_model)
        self.norm_h = nn.LayerNorm(d_model)
        self.pool_e = AttentivePooling(d_model)
        self.pool_h = AttentivePooling(d_model)
        fusion_dim = d_model * 8
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x_e2v, x_hc, mask=None, return_attention=False):
        e = self.e2v_proj(x_e2v)
        h = self.hc_proj(x_hc)
        key_padding_mask = mask == 0 if mask is not None else None

        e_att, attn_eh = self.e2v_to_hc(query=e, key=h, value=h, key_padding_mask=key_padding_mask, need_weights=True)
        h_att, attn_he = self.hc_to_e2v(query=h, key=e, value=e, key_padding_mask=key_padding_mask, need_weights=True)

        e_fused = self.norm_e(e + e_att)
        h_fused = self.norm_h(h + h_att)

        e_att_pool, e_pool_weights = self.pool_e(e_fused, mask)
        h_att_pool, h_pool_weights = self.pool_h(h_fused, mask)
        e_mean = masked_mean_pool(e_fused, mask)
        h_mean = masked_mean_pool(h_fused, mask)
        e_max = masked_max_pool(e_fused, mask)
        h_max = masked_max_pool(h_fused, mask)
        diff = torch.abs(e_att_pool - h_att_pool)
        prod = e_att_pool * h_att_pool

        fusion = torch.cat([e_att_pool, h_att_pool, e_mean, h_mean, e_max, h_max, diff, prod], dim=1)
        logits = self.classifier(fusion)

        if return_attention:
            return logits, {"attn_eh": attn_eh, "attn_he": attn_he, "pool_e": e_pool_weights, "pool_h": h_pool_weights}
        return logits



In [11]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for x_e2v, x_hc, mask, y_batch in loader:
        x_e2v = x_e2v.to(DEVICE)
        x_hc = x_hc.to(DEVICE)
        mask = mask.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(x_e2v, x_hc, mask)
            loss = criterion(logits, y_batch)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        total_loss += loss.item() * x_e2v.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_targets.extend(y_batch.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(np.array(all_targets), np.array(all_preds))
    return avg_loss, metrics


@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []

    for x_e2v, x_hc, mask, y_batch in loader:
        x_e2v = x_e2v.to(DEVICE)
        x_hc = x_hc.to(DEVICE)
        mask = mask.to(DEVICE)
        logits = model(x_e2v, x_hc, mask)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y_batch.numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return np.array(all_targets), np.array(all_preds), np.array(all_probs)



In [12]:
def train_eval_lodo_sequence_xattn(
    fold,
    seed,
    batch_size=8,
    lr=3e-4,
    weight_decay=1e-4,
    max_epochs=120,
    patience=15,
    d_model=128,
    num_heads=4,
    dropout=0.30,
):
    set_seed(seed)

    train_loader = make_sequence_loader(fold["X_e2v_train"], fold["X_hc_train"], fold["mask_train"], fold["y_train"], batch_size=batch_size, shuffle=True)
    val_loader = make_sequence_loader(fold["X_e2v_val"], fold["X_hc_val"], fold["mask_val"], fold["y_val"], batch_size=batch_size, shuffle=False)
    test_loader = make_sequence_loader(fold["X_e2v_test"], fold["X_hc_test"], fold["mask_test"], fold["y_test"], batch_size=batch_size, shuffle=False)

    model = SequenceCrossAttentionFusion(
        e2v_dim=fold["e2v_dim"],
        hc_dim=fold["hc_dim"],
        d_model=d_model,
        num_heads=num_heads,
        num_classes=len(LABELS),
        dropout=dropout,
    ).to(DEVICE)

    n_trainable = count_trainable_parameters(model)
    class_weights = compute_class_weights(fold["y_train"], n_classes=len(LABELS)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

    best_val_macro_f1 = -1.0
    best_epoch = -1
    best_state = None
    no_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss, train_metrics = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss, val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step(val_metrics["macro_f1"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        current = val_metrics["macro_f1"]
        if current > best_val_macro_f1:
            best_val_macro_f1 = current
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"[LODO Test={fold['test_dataset']} | seed={seed}] Epoch {epoch:03d} | val_macro_f1={val_metrics['macro_f1']:.4f} | best={best_val_macro_f1:.4f}")

        if no_improve >= patience:
            print(f"[LODO Test={fold['test_dataset']} | seed={seed}] Early stopping at epoch {epoch}. Best epoch={best_epoch}")
            break

    if best_state is None:
        raise RuntimeError("No best model state was saved. Check validation loop.")

    model.load_state_dict(best_state)
    y_val_true, y_val_pred, y_val_prob = predict_model(model, val_loader)
    y_test_true, y_test_pred, y_test_prob = predict_model(model, test_loader)
    val_metrics = compute_metrics(y_val_true, y_val_pred)
    test_metrics = compute_metrics(y_test_true, y_test_pred)

    run_dir = OUT_DIR / f"test_{fold['test_dataset']}" / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), run_dir / "sequence_cross_attention_lodo_model.pt")
    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

    with open(run_dir / "scaler_e2v_lodo_seq.pkl", "wb") as f:
        pickle.dump(fold["scaler_e2v"], f)
    with open(run_dir / "scaler_hc_lodo_seq.pkl", "wb") as f:
        pickle.dump(fold["scaler_hc"], f)

    training_config = {
        "model_name": "Sequence-level Cross-Attention",
        "test_dataset": fold["test_dataset"],
        "source_datasets": fold["source_datasets"],
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "batch_size": batch_size,
        "lr": lr,
        "weight_decay": weight_decay,
        "max_epochs": max_epochs,
        "patience": patience,
        "e2v_dim": fold["e2v_dim"],
        "hc_dim": fold["hc_dim"],
        "target_frames": fold["target_frames"],
        "d_model": d_model,
        "num_heads": num_heads,
        "dropout": dropout,
        "num_classes": len(LABELS),
        "trainable_parameters": n_trainable,
        "fusion_type": "temporal_bidirectional_cross_attention_lodo",
        "scaler_fit": "valid source-training frames only",
        "checkpoint_selection": "source-validation Macro-F1 per seed",
    }
    with open(run_dir / "training_config.json", "w") as f:
        json.dump(training_config, f, indent=2)

    pd.DataFrame([{ "model": "Sequence-level Cross-Attention", "test_dataset": fold["test_dataset"], "seed": seed, "split": "val", "best_epoch": best_epoch, **val_metrics }]).to_csv(run_dir / "val_metrics.csv", index=False)
    pd.DataFrame([{ "model": "Sequence-level Cross-Attention", "test_dataset": fold["test_dataset"], "seed": seed, "split": "test", "best_epoch": best_epoch, **test_metrics }]).to_csv(run_dir / "test_metrics.csv", index=False)

    make_report_df(y_val_true, y_val_pred).to_csv(run_dir / "val_classification_report.csv")
    make_report_df(y_test_true, y_test_pred).to_csv(run_dir / "test_classification_report.csv")
    save_confusion_matrix_csv(confusion_matrix(y_val_true, y_val_pred, labels=list(range(len(LABELS)))), run_dir / "val_confusion_matrix.csv")
    save_confusion_matrix_csv(confusion_matrix(y_test_true, y_test_pred, labels=list(range(len(LABELS)))), run_dir / "test_confusion_matrix.csv")

    pred_test_df = fold["meta_test"].copy()
    pred_test_df["y_true"] = y_test_true
    pred_test_df["y_pred"] = y_test_pred
    pred_test_df["true_label"] = [ID_TO_LABEL[i] for i in y_test_true]
    pred_test_df["pred_label"] = [ID_TO_LABEL[i] for i in y_test_pred]
    for i, label in enumerate(LABELS):
        pred_test_df[f"prob_{label}"] = y_test_prob[:, i]
    pred_test_df.to_csv(run_dir / "test_predictions.csv", index=False)

    row_val = {"model": "Sequence-level Cross-Attention", "test_dataset": fold["test_dataset"], "seed": seed, "split": "val", "best_epoch": best_epoch, "best_val_macro_f1": best_val_macro_f1, "trainable_parameters": n_trainable, **val_metrics}
    row_test = {"model": "Sequence-level Cross-Attention", "test_dataset": fold["test_dataset"], "seed": seed, "split": "test", "best_epoch": best_epoch, "best_val_macro_f1": best_val_macro_f1, "trainable_parameters": n_trainable, **test_metrics}

    return row_val, row_test



In [13]:
all_rows = []

for test_dataset in DATASETS:
    fold = make_lodo_sequence_fold(test_dataset)
    print("=" * 100)
    print(f"LODO SEQUENCE CROSS-ATTENTION | HELD-OUT: {test_dataset.upper()}")
    print(f"SOURCE DATASETS: {fold['source_datasets']}")
    print("=" * 100)

    for seed in SEEDS:
        print(f"\nTraining | test={test_dataset} | seed={seed}")
        row_val, row_test = train_eval_lodo_sequence_xattn(
            fold=fold,
            seed=seed,
            batch_size=8,
            lr=3e-4,
            weight_decay=1e-4,
            max_epochs=120,
            patience=15,
            d_model=128,
            num_heads=4,
            dropout=0.30,
        )
        all_rows.append(row_val)
        all_rows.append(row_test)
        print("VAL :", {k: round(v, 4) for k, v in row_val.items() if isinstance(v, float)})
        print("TEST:", {k: round(v, 4) for k, v in row_test.items() if isinstance(v, float)})

results = pd.DataFrame(all_rows)
results.to_csv(OUT_DIR / "lodo_sequence_xattn_all_seed_results.csv", index=False)

display(results)
print("Saved:", OUT_DIR / "lodo_sequence_xattn_all_seed_results.csv")



LODO SEQUENCE CROSS-ATTENTION | HELD-OUT: EMODB
SOURCE DATASETS: ['ravdess', 'resd']

Training | test=emodb | seed=42
[LODO Test=emodb | seed=42] Epoch 001 | val_macro_f1=0.7499 | best=0.7499
[LODO Test=emodb | seed=42] Epoch 010 | val_macro_f1=0.7714 | best=0.7855
[LODO Test=emodb | seed=42] Epoch 020 | val_macro_f1=0.7623 | best=0.7855
[LODO Test=emodb | seed=42] Early stopping at epoch 21. Best epoch=6
VAL : {'best_val_macro_f1': 0.7855, 'accuracy': 0.7873, 'macro_f1': 0.7855, 'weighted_f1': 0.7878, 'uar': 0.789}
TEST: {'best_val_macro_f1': 0.7855, 'accuracy': 0.8217, 'macro_f1': 0.8167, 'weighted_f1': 0.8176, 'uar': 0.8186}

Training | test=emodb | seed=123
[LODO Test=emodb | seed=123] Epoch 001 | val_macro_f1=0.7660 | best=0.7660
[LODO Test=emodb | seed=123] Epoch 010 | val_macro_f1=0.7509 | best=0.7946
[LODO Test=emodb | seed=123] Early stopping at epoch 17. Best epoch=2
VAL : {'best_val_macro_f1': 0.7946, 'accuracy': 0.7956, 'macro_f1': 0.7946, 'weighted_f1': 0.7956, 'uar': 0.79

,model,test_dataset,seed,split,best_epoch,best_val_macro_f1,trainable_parameters,accuracy,macro_f1,weighted_f1,uar
0,Sequence-level Cross-Attention,emodb,42,val,6,0.785512,913032,0.787293,0.785512,0.787833,0.789001
1,Sequence-level Cross-Attention,emodb,42,test,6,0.785512,913032,0.821727,0.816748,0.817641,0.818618
2,Sequence-level Cross-Attention,emodb,123,val,2,0.794614,913032,0.795580,0.794614,0.795609,0.795921
3,Sequence-level Cross-Attention,emodb,123,test,2,0.794614,913032,0.841226,0.838573,0.838454,0.839091
4,Sequence-level Cross-Attention,emodb,2024,val,6,0.780920,913032,0.781768,0.780920,0.782411,0.784919
5,Sequence-level Cross-Attention,emodb,2024,test,6,0.780920,913032,0.827298,0.821646,0.823028,0.825847
6,Sequence-level Cross-Attention,ravdess,42,val,2,0.664922,913032,0.665370,0.664922,0.662638,0.667900
7,Sequence-level Cross-Attention,ravdess,42,test,2,0.664922,913032,0.946023,0.943239,0.946423,0.948785
8,Sequence-level Cross-Attention,ravdess,123,val,1,0.677786,913032,0.673152,0.677786,0.672540,0.678909
9,Sequence-level Cross-Attention,ravdess,123,test,1,0.677786,913032,0.951705,0.949721,0.951868,0.953993


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_xattn_all_seed_results.csv


In [14]:
metrics = ["accuracy", "macro_f1", "weighted_f1", "uar"]
summary_rows = []

for test_dataset in DATASETS:
    for split in ["val", "test"]:
        sub = results[(results["test_dataset"] == test_dataset) & (results["split"] == split)]
        row = {
            "model": "Sequence-level Cross-Attention",
            "test_dataset": test_dataset,
            "split": split,
            "n_seeds": len(sub),
            "best_epoch_mean": sub["best_epoch"].mean(),
            "best_epoch_std": sub["best_epoch"].std(ddof=1),
            "trainable_parameters": sub["trainable_parameters"].iloc[0] if len(sub) else np.nan,
        }
        for metric in metrics:
            row[f"{metric}_mean"] = sub[metric].mean()
            row[f"{metric}_std"] = sub[metric].std(ddof=1)
        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "lodo_sequence_xattn_summary_mean_std.csv", index=False)

display(summary)
print("Saved:", OUT_DIR / "lodo_sequence_xattn_summary_mean_std.csv")



,model,test_dataset,split,n_seeds,best_epoch_mean,best_epoch_std,trainable_parameters,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,uar_mean,uar_std
0,Sequence-level Cross-Attention,emodb,val,3,4.666667,2.309401,913032,0.788214,0.006952,0.787015,0.006969,0.788618,0.006634,0.789947,0.005562
1,Sequence-level Cross-Attention,emodb,test,3,4.666667,2.309401,913032,0.830084,0.010043,0.825656,0.011452,0.826374,0.010802,0.827852,0.010383
2,Sequence-level Cross-Attention,ravdess,val,3,1.333333,0.577350,913032,0.669261,0.003891,0.670763,0.006513,0.667234,0.004989,0.672624,0.005668
3,Sequence-level Cross-Attention,ravdess,test,3,1.333333,0.577350,913032,0.946023,0.005682,0.943810,0.005648,0.946211,0.005766,0.949074,0.004781
4,Sequence-level Cross-Attention,resd,val,3,16.333333,6.806859,913032,0.890688,0.014597,0.886287,0.013849,0.890884,0.015138,0.898139,0.013630
5,Sequence-level Cross-Attention,resd,test,3,16.333333,6.806859,913032,0.587090,0.003475,0.582240,0.003175,0.587528,0.003435,0.576240,0.005287


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_xattn_summary_mean_std.csv


In [15]:
def mean_std_str(mean, std, scale=100):
    return f"{mean * scale:.2f} ± {std * scale:.2f}"


paper_rows = []
for test_dataset in DATASETS:
    sub = summary[(summary["test_dataset"] == test_dataset) & (summary["split"] == "test")].iloc[0]
    paper_rows.append({
        "Model": "Sequence-level Cross-Attention",
        "Held-out Test Dataset": test_dataset.upper(),
        "Accuracy": mean_std_str(sub["accuracy_mean"], sub["accuracy_std"]),
        "UAR": mean_std_str(sub["uar_mean"], sub["uar_std"]),
        "Macro-F1": mean_std_str(sub["macro_f1_mean"], sub["macro_f1_std"]),
        "Weighted-F1": mean_std_str(sub["weighted_f1_mean"], sub["weighted_f1_std"]),
    })

paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(OUT_DIR / "lodo_sequence_xattn_paper_table_test.csv", index=False)

display(paper_table)
print("Saved:", OUT_DIR / "lodo_sequence_xattn_paper_table_test.csv")



,Model,Held-out Test Dataset,Accuracy,UAR,Macro-F1,Weighted-F1
0,Sequence-level Cross-Attention,EMODB,83.01 ± 1.00,82.79 ± 1.04,82.57 ± 1.15,82.64 ± 1.08
1,Sequence-level Cross-Attention,RAVDESS,94.60 ± 0.57,94.91 ± 0.48,94.38 ± 0.56,94.62 ± 0.58
2,Sequence-level Cross-Attention,RESD,58.71 ± 0.35,57.62 ± 0.53,58.22 ± 0.32,58.75 ± 0.34


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_xattn_paper_table_test.csv


In [16]:
per_class_rows = []
for test_dataset in DATASETS:
    for seed in SEEDS:
        report_path = OUT_DIR / f"test_{test_dataset}" / f"seed_{seed}" / "test_classification_report.csv"
        report = pd.read_csv(report_path, index_col=0)
        for label in LABELS:
            per_class_rows.append({
                "model": "Sequence-level Cross-Attention",
                "heldout_dataset": test_dataset,
                "seed": seed,
                "class": label,
                "precision": report.loc[label, "precision"],
                "recall": report.loc[label, "recall"],
                "f1": report.loc[label, "f1-score"],
                "support": report.loc[label, "support"],
            })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(OUT_DIR / "lodo_sequence_xattn_per_class_all_seeds.csv", index=False)

per_class_summary = (
    per_class_df.groupby(["model", "heldout_dataset", "class"])
    .agg(
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        support_mean=("support", "mean"),
    )
    .reset_index()
)
per_class_summary.to_csv(OUT_DIR / "lodo_sequence_xattn_per_class_summary_mean_std.csv", index=False)

display(per_class_summary)
print("Saved:", OUT_DIR / "lodo_sequence_xattn_per_class_summary_mean_std.csv")



,model,heldout_dataset,class,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support_mean
0,Sequence-level Cross-Attention,emodb,angry,0.776673,0.019542,0.966667,0.004124,0.861196,0.010556,140.0
1,Sequence-level Cross-Attention,emodb,disgust,0.761662,0.037800,0.858491,0.000000,0.806857,0.020952,106.0
2,Sequence-level Cross-Attention,emodb,fear,0.948957,0.012709,0.604336,0.032857,0.738088,0.025293,123.0
3,Sequence-level Cross-Attention,emodb,happy,0.883449,0.015544,0.768362,0.017641,0.821713,0.007747,118.0
4,Sequence-level Cross-Attention,emodb,neutral,0.846134,0.044276,0.867925,0.034015,0.856022,0.022110,106.0
5,Sequence-level Cross-Attention,emodb,sad,0.841173,0.018004,0.901333,0.012220,0.870058,0.006467,125.0
6,Sequence-level Cross-Attention,ravdess,angry,0.945517,0.012234,0.991319,0.006014,0.967819,0.003594,192.0
7,Sequence-level Cross-Attention,ravdess,disgust,0.994624,0.000029,0.963542,0.005208,0.978831,0.002702,192.0
8,Sequence-level Cross-Attention,ravdess,fear,0.977407,0.009261,0.892361,0.026215,0.932724,0.010747,192.0
9,Sequence-level Cross-Attention,ravdess,happy,0.969768,0.019347,0.935764,0.006014,0.952352,0.006350,192.0


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_xattn_per_class_summary_mean_std.csv


In [17]:
paper_class_rows = []
for test_dataset in DATASETS:
    sub = per_class_summary[per_class_summary["heldout_dataset"] == test_dataset]
    row = {"Model": "Sequence-level Cross-Attention", "Held-out Dataset": test_dataset.upper()}
    for label in LABELS:
        label_row = sub[sub["class"] == label].iloc[0]
        row[label] = mean_std_str(label_row["f1_mean"], label_row["f1_std"])
    paper_class_rows.append(row)

paper_per_class_f1 = pd.DataFrame(paper_class_rows)
paper_per_class_f1.to_csv(OUT_DIR / "lodo_sequence_xattn_paper_table_per_class_f1.csv", index=False)

display(paper_per_class_f1)
print("Saved:", OUT_DIR / "lodo_sequence_xattn_paper_table_per_class_f1.csv")



,Model,Held-out Dataset,angry,disgust,fear,happy,neutral,sad
0,Sequence-level Cross-Attention,EMODB,86.12 ± 1.06,80.69 ± 2.10,73.81 ± 2.53,82.17 ± 0.77,85.60 ± 2.21,87.01 ± 0.65
1,Sequence-level Cross-Attention,RAVDESS,96.78 ± 0.36,97.88 ± 0.27,93.27 ± 1.07,95.24 ± 0.63,91.74 ± 0.90,91.37 ± 1.06
2,Sequence-level Cross-Attention,RESD,63.32 ± 0.45,57.69 ± 1.08,56.22 ± 0.41,65.87 ± 1.23,59.94 ± 1.77,46.32 ± 1.41


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/lodo_sequence_xattn_paper_table_per_class_f1.csv


In [18]:
UTTER_DIR = BASE_PROJECT / "results_lodo_utterance_fusion_plus_base"
SEQ_XATTN_DIR = BASE_PROJECT / "results_lodo_sequence_cross_attention_plus_base"

utter_summary_path = UTTER_DIR / "lodo_summary_mean_std.csv"
seq_summary_path = SEQ_XATTN_DIR / "lodo_sequence_xattn_summary_mean_std.csv"

if not utter_summary_path.exists():
    raise FileNotFoundError(f"Run notebook 10 first. Missing: {utter_summary_path}")

utter_summary = pd.read_csv(utter_summary_path)
seq_summary = pd.read_csv(seq_summary_path)

utter_test = utter_summary[utter_summary["split"] == "test"].copy()
seq_test = seq_summary[seq_summary["split"] == "test"].copy()
combined = pd.concat([utter_test, seq_test], ignore_index=True)

cols = [
    "model",
    "test_dataset",
    "accuracy_mean",
    "accuracy_std",
    "uar_mean",
    "uar_std",
    "macro_f1_mean",
    "macro_f1_std",
    "weighted_f1_mean",
    "weighted_f1_std",
]
combined = combined[cols]
combined.to_csv(SEQ_XATTN_DIR / "compare_lodo_all_models_test.csv", index=False)

display(combined)
print("Saved:", SEQ_XATTN_DIR / "compare_lodo_all_models_test.csv")



,model,test_dataset,accuracy_mean,accuracy_std,uar_mean,uar_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std
0,Handcrafted SVM-RBF,emodb,0.272981,0.000000e+00,0.258379,0.000000,0.197357,0.000000,0.205987,0.000000
1,Handcrafted SVM-RBF,ravdess,0.221591,0.000000e+00,0.203125,0.000000,0.141564,0.000000,0.154434,0.000000
2,Handcrafted SVM-RBF,resd,0.234558,3.399350e-17,0.245994,0.000000,0.225884,0.000000,0.223193,0.000000
3,Handcrafted MLP,emodb,0.285051,8.509890e-03,0.270249,0.011317,0.197985,0.014324,0.203113,0.015878
4,Handcrafted MLP,ravdess,0.259470,2.553308e-02,0.237847,0.023405,0.162702,0.031090,0.177494,0.033917
5,Handcrafted MLP,resd,0.237618,4.819284e-04,0.246064,0.003384,0.227669,0.007068,0.227536,0.008420
6,emotion2vec MLP,emodb,0.839833,5.021659e-03,0.838063,0.005197,0.835194,0.005231,0.836291,0.005525
7,emotion2vec MLP,ravdess,0.947285,3.942549e-03,0.949942,0.002790,0.945193,0.004245,0.947597,0.003834
8,emotion2vec MLP,resd,0.593767,2.683264e-03,0.584288,0.004125,0.593142,0.003418,0.596335,0.002645
9,Concat Fusion MLP,emodb,0.831941,9.782408e-03,0.830650,0.010662,0.826038,0.011932,0.827215,0.011249


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/compare_lodo_all_models_test.csv


In [19]:
paper_rows = []
for _, row in combined.iterrows():
    paper_rows.append({
        "Model": row["model"],
        "Held-out Test Dataset": row["test_dataset"].upper(),
        "Accuracy": mean_std_str(row["accuracy_mean"], row["accuracy_std"]),
        "UAR": mean_std_str(row["uar_mean"], row["uar_std"]),
        "Macro-F1": mean_std_str(row["macro_f1_mean"], row["macro_f1_std"]),
        "Weighted-F1": mean_std_str(row["weighted_f1_mean"], row["weighted_f1_std"]),
    })

paper_compare = pd.DataFrame(paper_rows)
paper_compare.to_csv(SEQ_XATTN_DIR / "compare_lodo_all_models_paper_table.csv", index=False)

display(paper_compare)
print("Saved:", SEQ_XATTN_DIR / "compare_lodo_all_models_paper_table.csv")



,Model,Held-out Test Dataset,Accuracy,UAR,Macro-F1,Weighted-F1
0,Handcrafted SVM-RBF,EMODB,27.30 ± 0.00,25.84 ± 0.00,19.74 ± 0.00,20.60 ± 0.00
1,Handcrafted SVM-RBF,RAVDESS,22.16 ± 0.00,20.31 ± 0.00,14.16 ± 0.00,15.44 ± 0.00
2,Handcrafted SVM-RBF,RESD,23.46 ± 0.00,24.60 ± 0.00,22.59 ± 0.00,22.32 ± 0.00
3,Handcrafted MLP,EMODB,28.51 ± 0.85,27.02 ± 1.13,19.80 ± 1.43,20.31 ± 1.59
4,Handcrafted MLP,RAVDESS,25.95 ± 2.55,23.78 ± 2.34,16.27 ± 3.11,17.75 ± 3.39
5,Handcrafted MLP,RESD,23.76 ± 0.05,24.61 ± 0.34,22.77 ± 0.71,22.75 ± 0.84
6,emotion2vec MLP,EMODB,83.98 ± 0.50,83.81 ± 0.52,83.52 ± 0.52,83.63 ± 0.55
7,emotion2vec MLP,RAVDESS,94.73 ± 0.39,94.99 ± 0.28,94.52 ± 0.42,94.76 ± 0.38
8,emotion2vec MLP,RESD,59.38 ± 0.27,58.43 ± 0.41,59.31 ± 0.34,59.63 ± 0.26
9,Concat Fusion MLP,EMODB,83.19 ± 0.98,83.07 ± 1.07,82.60 ± 1.19,82.72 ± 1.12


Saved: /content/drive/MyDrive/New Jurnal Cross/results_lodo_sequence_cross_attention_plus_base/compare_lodo_all_models_paper_table.csv
